# M8 — UniXcoder + SVM Linéaire

Ce modèle utilise UniXcoder pour représenter le code source des solutions sous forme d'embeddings, puis un SVM linéaire pour prédire les huit tags de chaque exercice. L'encodeur préentraîné est utilisé sans fine-tuning, et les représentations des différents blocs sont agrégées lorsque le code dépasse la longueur maximale du modèle. Après l'entraînement, plusieurs configurations d'hyperparamètres sont comparées sur l'ensemble de validation afin de sélectionner la meilleure selon le F1 macro. Les seuils de décision sont ensuite ajustés pour chaque tag, puis le modèle est évalué et sauvegardé. L'ensemble de test reste réservé à l'évaluation finale.

## 1. Charger et contrôler les données

Seul le code source de la solution sert d’entrée. Les énoncés, identifiants et annotations ne sont jamais transmis au vectoriseur. 

In [2]:
import hashlib
import json
import os
import platform
import warnings
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import joblib
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    precision_recall_fscore_support,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits
from transformers import RobertaConfig, RobertaModel, RobertaTokenizer

racine = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "entrainement.jsonl").is_file()), None
)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

In [3]:
dossier_modele = racine / "models" / "m8_unixcoder_linear_svm"
dossier_modele.mkdir(parents=True, exist_ok=True)
graine = 42
np.random.seed(graine)
torch.manual_seed(graine)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
champ_code = "source_code"
versions = {nom: version(nom) for nom in
            ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "threadpoolctl",
             "torch", "transformers", "tokenizers", "huggingface-hub", "safetensors"]}
empreintes = {}

def charger_ensemble(nom):
    chemin = racine / "data" / f"{nom}.jsonl"
    contenu = chemin.read_bytes()
    empreintes[nom] = hashlib.sha256(contenu).hexdigest()
    return pd.DataFrame([json.loads(ligne) for ligne in contenu.decode("utf-8").splitlines()
                         if ligne.strip()])

entrainement = charger_ensemble("entrainement")
validation = charger_ensemble("validation")
for donnees in [entrainement, validation]:
    assert donnees[colonnes_cibles].isin([0, 1]).all().all(), "Cibles non binaires."
    assert champ_code in donnees.columns, "Le champ source_code est absent."
    assert donnees[champ_code].map(lambda code: isinstance(code, str) and bool(code.strip())).all(), \
        "Un code source est absent ou vide."
    assert not donnees[champ_code].duplicated().any(), "Des codes identiques sont présents."
assert set(entrainement[champ_code]).isdisjoint(validation[champ_code]), \
    "Un même code apparaît dans l’entraînement et la validation."
y_train = entrainement[colonnes_cibles].to_numpy(dtype=np.int64)
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)
for champ in ["fichier_source", "src_uid", "code_uid"]:
    assert not entrainement[champ].isna().any() and not validation[champ].isna().any()
    assert not entrainement[champ].duplicated().any()
    assert not validation[champ].duplicated().any()
    assert set(entrainement[champ]).isdisjoint(validation[champ]), f"Fuite : {champ}"
for cibles in [y_train, y_validation]:
    assert ((cibles.sum(axis=0) > 0) & (cibles.sum(axis=0) < len(cibles))).all()

warnings.filterwarnings("error", category=ConvergenceWarning)
print(f"{len(entrainement)} exercices d’entraînement et {len(validation)} de validation.")

copies_train = entrainement[champ_code].str.replace(r"\s+", "", regex=True)
copies_validation = validation[champ_code].str.replace(r"\s+", "", regex=True)
display(pd.DataFrame([
    {"controle": "Codes répétés hors espaces — entraînement",
     "nombre": int(copies_train.duplicated().sum())},
    {"controle": "Codes répétés hors espaces — validation",
     "nombre": int(copies_validation.duplicated().sum())},
    {"controle": "Codes communs hors espaces — entraînement / validation",
     "nombre": len(set(copies_train) & set(copies_validation))},
]))

3470 exercices d’entraînement et 739 de validation.


,controle,nombre
0,Codes répétés hors espaces — entraînement,0
1,Codes répétés hors espaces — validation,0
2,Codes communs hors espaces — entraînement / va...,0


## 2. Construire les embeddings UniXcoder

Le modèle préentraîné UniXcoder est utilisé sur CPU, sans fine-tuning, pour transformer le code source des solutions en embeddings de 768 dimensions. Le code est conservé tel quel, sans nettoyage. Pour respecter la limite de 512 tokens du modèle, les codes longs sont découpés en plusieurs blocs, sans perte de contenu. Chaque bloc est encodé séparément, puis les embeddings sont regroupés par moyenne pondérée selon leur longueur et normalisés en L2.

In [ ]:
nom_encodeur = "microsoft/unixcoder-base"
revision_encodeur = "5604afdc964f6c53782a6813140ade5216b99006"
parametres_encodage = {"taille_bloc": 508, "taille_lot": 8}
tokenizer = RobertaTokenizer.from_pretrained(nom_encodeur, revision=revision_encodeur)
configuration_encodeur = RobertaConfig.from_pretrained(nom_encodeur, revision=revision_encodeur)
configuration_encodeur.is_decoder = False
configuration_encodeur.use_cache = False
encodeur = RobertaModel.from_pretrained(
    nom_encodeur, revision=revision_encodeur, config=configuration_encodeur,
    attn_implementation="eager",
).to("cpu")
encodeur.requires_grad_(False)
encodeur.eval()
assert encodeur.config.hidden_size == 768
assert encodeur.config.max_position_embeddings >= 514
assert tokenizer.convert_tokens_to_ids("<encoder-only>") != tokenizer.unk_token_id
assert tokenizer.pad_token_id == encodeur.config.pad_token_id
codes_train = entrainement[champ_code]
codes_validation = validation[champ_code]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8020.01it/s]


In [5]:
def decouper_codes(codes, tokenizer, taille_bloc=508):
    if not 1 <= taille_bloc <= 508:
        raise ValueError("Les blocs doivent contenir entre 1 et 508 tokens de code.")
    if any(not isinstance(code, str) or not code.strip() for code in codes):
        raise ValueError("Chaque solution doit contenir un code non vide.")
    contenus = tokenizer(list(codes), add_special_tokens=False,
                         truncation=False, padding=False, verbose=False)["input_ids"]
    if not contenus or any(not contenu for contenu in contenus):
        raise ValueError("Un code ne contient aucun token.")
    prefixe = [tokenizer.cls_token_id,
               tokenizer.convert_tokens_to_ids("<encoder-only>"), tokenizer.sep_token_id]
    blocs, proprietaires, poids = [], [], []
    for indice, contenu in enumerate(contenus):
        for debut in range(0, len(contenu), taille_bloc):
            morceau = contenu[debut:debut + taille_bloc]
            ids = prefixe + morceau + [tokenizer.sep_token_id]
            blocs.append({"input_ids": ids, "attention_mask": [1] * len(ids)})
            proprietaires.append(indice)
            poids.append(len(morceau))
    longueurs = np.array([len(contenu) for contenu in contenus])
    proprietaires, poids = np.array(proprietaires), np.array(poids)
    np.testing.assert_array_equal(
        np.bincount(proprietaires, weights=poids, minlength=len(contenus)), longueurs
    )
    assert all(len(bloc["input_ids"]) <= 512 for bloc in blocs)
    return blocs, proprietaires, poids, longueurs

def encoder_codes(codes, encodeur, tokenizer, taille_bloc=508, taille_lot=8):
    if taille_lot < 1:
        raise ValueError("La taille des lots doit être positive.")
    blocs, proprietaires, poids, longueurs = decouper_codes(codes, tokenizer, taille_bloc)
    ordre = np.argsort([len(bloc["input_ids"]) for bloc in blocs], kind="stable")
    blocs = [blocs[indice] for indice in ordre]
    proprietaires, poids = proprietaires[ordre], poids[ordre]
    vecteurs = np.zeros((len(longueurs), encodeur.config.hidden_size), dtype=np.float64)
    encodeur.eval()
    appareil = next(encodeur.parameters()).device
    with torch.inference_mode():
        for debut in range(0, len(blocs), taille_lot):
            fin = debut + taille_lot
            lot = tokenizer.pad(blocs[debut:fin], padding=True, return_tensors="pt")
            ids = lot["input_ids"].to(appareil)
            masque = lot["attention_mask"].to(appareil)
            sorties = encodeur(input_ids=ids, attention_mask=masque).last_hidden_state
            embeddings = (sorties * masque.unsqueeze(-1)).sum(1) / masque.sum(1, keepdim=True)
            embeddings = embeddings.cpu().numpy()
            np.add.at(vecteurs, proprietaires[debut:fin], embeddings * poids[debut:fin, None])
    vecteurs /= longueurs[:, None]
    normes = np.linalg.norm(vecteurs, axis=1, keepdims=True)
    assert np.isfinite(vecteurs).all() and (normes > 0).all()
    vecteurs = (vecteurs / normes).astype(np.float32)
    return vecteurs, longueurs, np.bincount(proprietaires, minlength=len(longueurs))

debut = perf_counter()
X_train, longueurs_train, blocs_train = encoder_codes(
    codes_train, encodeur, tokenizer, **parametres_encodage
)
duree_train = perf_counter() - debut
debut = perf_counter()
X_validation, longueurs_validation, blocs_validation = encoder_codes(
    codes_validation, encodeur, tokenizer, **parametres_encodage
)
duree_validation = perf_counter() - debut
duree_encodage = duree_train + duree_validation
assert X_train.shape == (len(entrainement), 768)
assert X_validation.shape == (len(validation), 768)
for matrice in [X_train, X_validation]:
    np.testing.assert_allclose(np.linalg.norm(matrice, axis=1), 1, atol=1e-6)
assert not any(parametre.requires_grad for parametre in encodeur.parameters())
display(pd.DataFrame([
    {"ensemble": nom, "exercices": len(longueurs),
     "tokens_medians": float(np.median(longueurs)), "tokens_maximum": int(longueurs.max()),
     "depassent_512_tokens": int((longueurs + 4 > 512).sum()),
     "pourcentage_codes_decoupes": 100 * float((longueurs + 4 > 512).mean()),
     "blocs_encodes": int(blocs.sum()), "temps_encodage_s": duree}
    for nom, longueurs, blocs, duree in [
        ("Entraînement", longueurs_train, blocs_train, duree_train),
        ("Validation", longueurs_validation, blocs_validation, duree_validation)]
]).round(2))

,ensemble,exercices,tokens_medians,tokens_maximum,depassent_512_tokens,pourcentage_codes_decoupes,blocs_encodes,temps_encodage_s
0,Entraînement,3470,282.0,19553,905,26.08,5234,1274.36
1,Validation,739,292.0,10023,192,25.98,1108,271.99


## 3. Comparer six configurations

Huit classifieurs binaires sont entraînés via `OneVsRestClassifier`.
Les six couples `C ∈ {0.1, 1, 10}` et `class_weight ∈ {None, balanced}`
sont comparés avec le seuil par défaut. Le F1 macro donne le même poids
aux huit tags, le F1 micro complète la lecture globale.

In [6]:
def evaluer_predictions(reels, predictions):
    return {
        "f1_macro": float(f1_score(reels, predictions, average="macro", zero_division=0)),
        "f1_micro": float(f1_score(reels, predictions, average="micro", zero_division=0)),
        "hamming_loss": float(hamming_loss(reels, predictions)),
        "exact_match_ratio": float(accuracy_score(reels, predictions)),
    }

In [7]:
seuil_defaut = 0.0

def creer_classifieur(C, ponderation):
    estimateur = LinearSVC(C=C, class_weight=ponderation, dual="auto", max_iter=10000, random_state=graine)
    return OneVsRestClassifier(estimateur, n_jobs=1)

def calculer_scores(classifieur, matrice):
    return classifieur.decision_function(matrice)

experiences = []
classifieurs = {}
with threadpool_limits(limits=1):
    for C in [0.1, 1.0, 10.0]:
        for nom_ponderation, ponderation in [("aucune", None), ("equilibree", "balanced")]:
            cle = f"C{C}_{nom_ponderation}"
            classifieur = creer_classifieur(C, ponderation)
            debut = perf_counter()
            classifieur.fit(X_train, y_train)
            duree = perf_counter() - debut
            scores = calculer_scores(classifieur, X_validation)
            predictions = (scores >= seuil_defaut).astype(int)
            classifieurs[cle] = classifieur
            experiences.append({"experience": cle, "C": C, "class_weight": nom_ponderation,
                                 "temps_entrainement_s": duree,
                                 **evaluer_predictions(y_validation, predictions)})

recherche = pd.DataFrame(experiences).sort_values(
    ["f1_macro", "C"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
configuration_retenue = recherche.iloc[0]
classifieur_final = classifieurs[configuration_retenue["experience"]]
scores_validation = calculer_scores(classifieur_final, X_validation)
display(recherche.round(4))
print("Configuration retenue :", configuration_retenue["experience"])

,experience,C,class_weight,temps_entrainement_s,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,C10.0_aucune,10.0,aucune,2.7667,0.3634,0.4396,0.0785,0.4980
1,C1.0_equilibree,1.0,equilibree,2.3804,0.3555,0.4403,0.1363,0.3126
2,C10.0_equilibree,10.0,equilibree,4.3267,0.3483,0.4427,0.1209,0.3681
3,C0.1_equilibree,0.1,equilibree,1.5309,0.3427,0.4146,0.1576,0.2449
4,C1.0_aucune,1.0,aucune,1.7265,0.2838,0.4060,0.0743,0.5169
5,C0.1_aucune,0.1,aucune,1.2767,0.1396,0.2950,0.0768,0.5034


Configuration retenue : C10.0_aucune


## 4. Ajuster les huit seuils

Le classifieur retenu est conservé sans réentraînement. Pour chaque tag, choisir
le seuil donnant le meilleur F1 parmi une grille de quantiles des scores de validation
et le seuil par défaut (0.0).

In [8]:
quantiles_seuils = [0.50, 0.70, 0.80, 0.85, 0.90, 0.925,
                   0.95, 0.965, 0.975, 0.985, 0.99, 0.995]
seuils = []
for indice in range(len(tags_cibles)):
    candidats = np.unique(np.append(
        np.quantile(scores_validation[:, indice], quantiles_seuils), seuil_defaut
    ))
    evaluations = [
        (f1_score(y_validation[:, indice], scores_validation[:, indice] >= seuil,
                  zero_division=0), float(seuil)) for seuil in candidats
    ]
    meilleur = min(evaluations, key=lambda r: (-r[0], abs(r[1] - seuil_defaut), r[1]))
    seuils.append(meilleur[1])
seuils = np.array(seuils)

## 5. Lire les résultats de validation

Comparer les seuils par défaut et ajustés, puis examiner chaque tag avec son support,
sa précision, son rappel et son F1. 

In [9]:
lignes_globales, lignes_tags = [], []
for nom, valeur in [("Aucun tag", 0), ("Tous les tags", 1)]:
    predictions = np.full_like(y_validation, valeur)
    lignes_globales.append({"regle": nom, **evaluer_predictions(
        y_validation, predictions
    )})
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = (scores_validation >= seuils_regle).astype(int)
    lignes_globales.append({"regle": regle,
                           **evaluer_predictions(y_validation, predictions)})
    precisions, rappels, f1, supports = precision_recall_fscore_support(
        y_validation, predictions, average=None, zero_division=0
    )
    for i, tag in enumerate(tags_cibles):
        lignes_tags.append({
            "regle": regle, "tag": tag, "support": int(supports[i]),
            "seuil": float(seuils_regle[i]), "precision": precisions[i],
            "rappel": rappels[i], "f1": f1[i],
        })
bilan = pd.DataFrame(lignes_globales)
par_tag = pd.DataFrame(lignes_tags)
display(bilan.round(4))
display(par_tag.loc[par_tag["regle"].eq("Ajustés")].round(4).reset_index(drop=True))

sans_cible = y_validation.sum(axis=1) == 0
controle_sans_cible = []
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = scores_validation[sans_cible] >= seuils_regle
    avec_prediction = predictions.any(axis=1)
    controle_sans_cible.append({
        "regle": regle,
        "exercices_sans_cible": int(sans_cible.sum()),
        "aucun_tag_predit": int((~avec_prediction).sum()),
        "au_moins_un_tag_predit": int(avec_prediction.sum()),
    })
print("Contrôle des exercices sans tag cible (décomptes) :")
display(pd.DataFrame(controle_sans_cible))

,regle,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,Aucun tag,0.0000,0.0000,0.0864,0.4587
1,Tous les tags,0.1500,0.1591,0.9136,0.0000
2,Défaut,0.3634,0.4396,0.0785,0.4980
3,Ajustés,0.4001,0.4835,0.1033,0.3884


,regle,tag,support,seuil,precision,rappel,f1
0,Ajustés,math,211,-0.5224,0.4054,0.7109,0.5164
1,Ajustés,graphs,81,-0.4202,0.4505,0.6173,0.5208
2,Ajustés,strings,63,-0.3930,0.4324,0.5079,0.4672
3,Ajustés,number theory,52,-0.4360,0.3514,0.5000,0.4127
4,Ajustés,trees,49,0.2859,0.7895,0.3061,0.4412
5,Ajustés,geometry,25,-0.2384,0.4737,0.3600,0.4091
6,Ajustés,games,16,-0.2987,0.7500,0.1875,0.3000
7,Ajustés,probabilities,14,0.0000,1.0000,0.0714,0.1333


Contrôle des exercices sans tag cible (décomptes) :


,regle,exercices_sans_cible,aucun_tag_predit,au_moins_un_tag_predit
0,Défaut,339,270,69
1,Ajustés,339,147,192


## 6. Sauvegarder le modèle final et les résultats

Un seul artefact contient UniXcoder avec ses poids et son tokenizer, les huit
classifieurs, les seuils, l’ordre des tags et les paramètres d’encodage. Les trois CSV conservent la recherche et les métriques. Le modèle sauvegardé reste celui appris sur l’entraînement, les seuils ont été réglés sur la validation.

In [10]:
modele_final = {
    "modele": "M8", "encodeur": encodeur, "tokenizer": tokenizer, "classifieur": classifieur_final,
    "seuils": seuils, "tags": tags_cibles, "champ_code": champ_code,
    "type_scores": "marges_de_decision",
    "pretraitement": {"code_inchange": True, "html_unescape": False},
    "nom_encodeur": nom_encodeur, "revision_encodeur": revision_encodeur,
    "encodage": {**parametres_encodage, "longueur_max": 512, "dimension": 768,
                 "mode": "<encoder-only>", "attention": "bidirectionnelle",
                 "pooling": "moyenne_tokens_hors_padding",
                 "agregation": "moyenne_ponderee_par_tokens", "normalisation": "l2"},
    "temps_encodage_s": {"entrainement": duree_train, "validation": duree_validation},
    "materiel": {"appareil": "cpu", "architecture": platform.machine(), "threads_torch": 4},
    "configuration": {"C": float(configuration_retenue["C"]),
                      "class_weight": None if configuration_retenue["class_weight"] == "aucune"
                      else "balanced"},
    "graine": graine, "versions": versions, "python": platform.python_version(),
    "empreintes_donnees": empreintes,
}
_ = joblib.dump(modele_final, dossier_modele / "modele.joblib", compress=3)
recherche.to_csv(dossier_modele / "recherche_hyperparametres.csv", index=False)
bilan.to_csv(dossier_modele / "metriques_validation.csv", index=False)
par_tag.to_csv(dossier_modele / "metriques_par_tag_validation.csv", index=False)